# Deliverable 2 - Feature Engineering

Data preparation for classification.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'wine_quality_project').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'wine_quality_project'
elif PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
print('PROJECT_ROOT =', PROJECT_ROOT)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
red = pd.read_csv(PROJECT_ROOT / 'data/raw/winequality-red.csv', sep=';')
white = pd.read_csv(PROJECT_ROOT / 'data/raw/winequality-white.csv', sep=';')
red['wine_type'] = 'red'
white['wine_type'] = 'white'
wine = pd.concat([red, white], ignore_index=True)
wine['quality_group'] = wine['quality'].apply(lambda q: 'low quality' if q <= 5 else ('medium quality' if q == 6 else 'high quality'))
wine.to_csv(PROJECT_ROOT / 'data/processed/wine_quality_combined_with_target.csv', index=False)
wine.head()


PROJECT_ROOT = /Users/oscarmagniez/Desktop/Python/vin_project/wine_quality_project


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type,quality_group
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red,low quality
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red,low quality
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red,low quality
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red,medium quality
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red,low quality


In [2]:
features = ['fixed acidity','volatile acidity','citric acid','residual sugar','chlorides','free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','alcohol','wine_type']
numeric_features = [c for c in features if c != 'wine_type']
X = wine[features]
y = wine['quality_group']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(5197, 12) (1300, 12)


In [3]:
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('encoder', encoder)]), ['wine_type']),
])
X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr = preprocessor.transform(X_test)
feature_names = [n.replace('num__','').replace('cat__','') for n in preprocessor.get_feature_names_out()]
X_train_engineered = pd.DataFrame(X_train_arr, columns=feature_names)
X_test_engineered = pd.DataFrame(X_test_arr, columns=feature_names)
X_train_engineered.to_csv(PROJECT_ROOT / 'data/processed/X_train_engineered.csv', index=False)
X_test_engineered.to_csv(PROJECT_ROOT / 'data/processed/X_test_engineered.csv', index=False)
y_train.reset_index(drop=True).to_frame('quality_group').to_csv(PROJECT_ROOT / 'data/processed/y_train.csv', index=False)
y_test.reset_index(drop=True).to_frame('quality_group').to_csv(PROJECT_ROOT / 'data/processed/y_test.csv', index=False)
X_train_engineered.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,wine_type_red,wine_type_white
0,-0.473365,0.312756,0.489564,1.349979,0.038654,1.146498,1.844836,1.275912,0.253136,-0.134873,-1.333447,0.0,1.0
1,1.069730,0.924012,1.314887,-0.724595,10.708489,-0.822870,-0.958317,1.071721,-1.174245,4.417558,-1.249488,1.0,0.0
2,-1.167758,-0.481877,-0.542090,2.586342,0.097119,-0.428996,0.134736,1.014816,0.501376,-0.203850,-0.325946,0.0,1.0
3,3.307218,0.435007,1.383664,-0.515042,0.711000,-0.091390,-1.081726,1.808146,0.253136,2.417247,0.429679,1.0,0.0
4,3.461528,0.679509,2.140211,-0.682684,0.506373,-1.329279,-1.645883,1.687641,-0.057164,1.106699,0.345720,1.0,0.0


The split is done before preprocessing and `quality` / `quality_group` are not included in the input features.